In [1]:
import pandas as pd
import os
import re

df_14 = pd.read_csv("RCFL_Microdati_2014_Primo_trimestre.txt",sep="\t",low_memory=False)
df_15 = pd.read_csv("RCFL_Microdati_2015_Primo_trimestre.txt",sep="\t",low_memory=False)
df_16 = pd.read_csv("RCFL_Microdati_2016_Primo_trimestre.txt",sep="\t",low_memory=False)
df_17 = pd.read_csv("RCFL_Microdati_2017_Primo_trimestre.txt",sep="\t",low_memory=False)
df_18 = pd.read_csv("RCFL_Microdati_2018_Primo_trimestre.txt",sep="\t",low_memory=False)
df_19 = pd.read_csv("RCFL_Microdati_2019_Primo_trimestre.txt",sep="\t",low_memory=False)
df_20 = pd.read_csv("RCFL_Microdati_2020_Primo_trimestre.txt",sep="\t",low_memory=False)
df_21 = pd.read_csv("RCFL_Microdati_2021_Primo_trimestre.txt",sep="\t",low_memory=False)
df_22 = pd.read_csv("RCFL_Microdati_2022_Primo_trimestre.txt",sep="\t",low_memory=False)
df_23 = pd.read_csv("RCFL_Microdati_2023_Primo_trimestre.txt",sep="\t",low_memory=False)


In [2]:
df_list = [df_14,df_15,df_16,df_17,df_18,df_19,df_20,df_21,df_22,df_23]
years_t = list(range(2014,2024))

print(df_21.shape)
print(df_21.columns[:50])

(120351, 228)
Index(['ANNO', 'TRIM', 'WAVQUA', 'QSF11', 'QSF17', 'QSF20', 'QSF26', 'QSF30',
       'QSF32', 'QB01', 'QB02', 'QB03', 'QB04', 'QB05', 'QB06', 'QB07', 'QB08',
       'QB09', 'Qbtemp', 'QB10', 'QB11', 'QC04', 'QC05BIS', 'QC06', 'QC11',
       'QC12', 'QC13', 'QC14', 'QC15', 'QC16', 'QC17', 'QC18', 'QC19', 'QC20',
       'QC21', 'QC22', 'QC25', 'QC26', 'QC27', 'QC28', 'QC29', 'QC30', 'QC31A',
       'QC34', 'QC35A', 'QC36', 'QC38_1', 'QC38_2', 'QC38_3', 'QC42'],
      dtype='str')


In [3]:
print(years_t)

[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


In [4]:
df_21.head()

,ANNO,TRIM,WAVQUA,QSF11,QSF17,QSF20,QSF26,QSF30,QSF32,QB01,...,QH04MCR,QH33MCR,QSF04MCR,QSF24MCR,QSF27MCR,TFMCR,MCRFAM,MCRIND,AMATRIMCR,INCDEC
0,2021,1,4,2,2,2,1,002,,2,...,2,,7,1,,,,155,1,
1,2021,1,2,1,2,2,1,002,,1,...,5,,7,3,,,,484,1,01
2,2021,1,4,2,2,2,1,002,,2,...,5,,7,3,,,,630,3,
3,2021,1,4,2,1,1,1,,,2,...,6,,7,,,,,727,5,
4,2021,1,2,2,1,2,1,002,,2,...,,,7,3,,,,970,,


In [5]:
df_21.head(5)["QC52"]

0       
1    003
2       
3       
4       
Name: QC52, dtype: str

In [6]:
df_21.head(22)["REGMCR"]

0     19
1     14
2     13
3      8
4      3
5     19
6     14
7      5
8      6
9     16
10     3
11     5
12     3
13     3
14     1
15     3
16    18
17     4
18    15
19     3
20     9
21    19
Name: REGMCR, dtype: int64

In [7]:
region_dict = {
    "01": "Piemonte/Valle d'Aosta",
    "03": "Lombardia",
    "04": "Trentino-Alto Adige",
    "05": "Veneto",
    "06": "Friuli-Venezia Giulia",
    "07": "Liguria",
    "08": "Emilia-Romagna",
    "09": "Toscana",
    "10": "Umbria",
    "11": "Le Marche",
    "12": "Lazio",
    "13": "Abruzzo",
    "14": "Molise",
    "15": "Campania",
    "16": "Puglia",
    "17": "Basilicata",
    "18": "Calabria",
    "19": "Sicilia",
    "20": "Sardegna"}

In [8]:
def get_remote_work_share(df):

    df_emp = df[df["QC52"].isin(["001","002","003"])]

    regional_remote = (
        df_emp
        .groupby("REGMCR")
        .apply(lambda x:
            (x["QC52"].isin(["001","002"]).astype(int) * x["COEF_CCP"]).sum()
            / x["COEF_CCP"].sum()
        )
    )
    return regional_remote

"""
for df in df_list:
    regional_remote = get_remote_work_share(df)
"""

regional_remote = get_remote_work_share(df_21)

print(regional_remote.sort_values())

REGMCR
18    0.123477
17    0.128606
19    0.135014
16    0.150211
11    0.150766
14    0.155325
13    0.156716
15    0.158216
5     0.166309
20    0.171012
10    0.176935
6     0.181692
9     0.185753
1     0.187549
4     0.192762
8     0.210628
7     0.218108
3     0.221966
12    0.269722
dtype: float64


In [ ]:
def build_remote_work_panel_from_dfs(dfs, years, output_csv='remote_work_panel.csv'):
    all_years = []
    for df, year in zip(dfs, years):
        if "REG" in df.columns: #Rename REG to REGMCR if needed
            df["REG"] = df["REG"].astype(str).str.zfill(2)  # Ensure codes are two digits
            df.rename(columns={"REG": "REGMCR"}, inplace=True)

        if 'QC52' not in df.columns:
            print(f"QC52 missing in year {year}; filling remote share with 0 for all regions.")
            missing_year = pd.DataFrame({
                'REGMCR': [str(k).zfill(2) for k in region_dict.keys()],
                'remote': [0.0] * len(region_dict)
            })
            missing_year['year'] = year
            all_years.append(missing_year)
            continue

        temp = df[['QC52','REGMCR','COND3','COEF_CCP']].copy()
        #   keep only employed persons
        temp = temp[temp['COND3'] == 1]   # check exact code meaning in readme
        # drop missing QC52
        temp = temp[temp['QC52'].isin(['001','002','003'])]  # 001 = yes, 002 = sometimes, 003 = no
        temp['remote'] = temp['QC52'].isin(['001','002']).astype(int)
        regional_share = (
            temp
            .groupby('REGMCR')
            .apply(lambda g: (g['remote'] * g['COEF_CCP']).sum() / g['COEF_CCP'].sum())
            .reset_index()
            .rename(columns={0: 'remote'}))
        regional_share['year'] = year
        all_years.append(regional_share)
    panel = pd.concat(all_years, ignore_index=True)
    #Replace region codes with names
    panel['REGMCR'] = panel['REGMCR'].astype(str).str.zfill(2)  # Ensure codes are two digits
    panel['REGMCR'] = panel['REGMCR'].map(region_dict)
    panel_pivot = panel.pivot(index='REGMCR', columns='year', values='remote').reset_index()
    panel_pivot[2020] = (panel_pivot[2021] + 0.045) #0.045 from 2021
    panel_pivot[2019] = (panel_pivot[2021] -0.101) #0.101 from 2021
    panel_pivot[2018] = panel_pivot[2019] * (1-0.2)
    panel_pivot[0:] = panel_pivot[0:].round(4)
    panel_pivot.to_csv(output_csv, index=False)
    return panel_pivot

n2_remote_work_panel = build_remote_work_panel_from_dfs(
    dfs= df_list,
    years= years_t,
    output_csv='n2_remote_work_panel.csv'
)

QC52 missing in year 2014; filling remote share with 0 for all regions.
QC52 missing in year 2015; filling remote share with 0 for all regions.
QC52 missing in year 2016; filling remote share with 0 for all regions.
QC52 missing in year 2017; filling remote share with 0 for all regions.
QC52 missing in year 2018; filling remote share with 0 for all regions.
QC52 missing in year 2019; filling remote share with 0 for all regions.
QC52 missing in year 2020; filling remote share with 0 for all regions.


In [ ]:
def build_var_panel_from_dfs(dfs, years, var, codes, var_codes, output_csv='remote_work_panel.csv'):
    all_years = []
    for df, year in zip(dfs, years):
        if "REG" in df.columns: #Rename REG to REGMCR if needed
            df["REG"] = df["REG"].astype(str).str.zfill(2)  # Ensure codes are two digits
            df.rename(columns={"REG": "REGMCR"}, inplace=True)
        if var not in df.columns:
            missing_year = pd.DataFrame({
                'REGMCR': [str(k).zfill(2) for k in region_dict.keys()],
                'remote': [float('nan')] * len(region_dict),
            })
            missing_year['year'] = year
            all_years.append(missing_year)
            continue

        temp = df[[var,'REGMCR','COND3','COEF_CCP']].copy()
        #   keep only employed persons
        temp = temp[temp['COND3'] == 1]   # check exact code meaning in readme
        # drop missing and invalid values
        temp = temp[temp[var].isin(codes)]
        temp['remote'] = temp[var].isin(var_codes).astype(int)
        regional_share = (
            temp
            .groupby('REGMCR')
            .apply(lambda g: (g['remote'] * g['COEF_CCP']).sum() / g['COEF_CCP'].sum())
            .reset_index()
            .rename(columns={0: 'remote'}))
        regional_share['year'] = year
        all_years.append(regional_share)
    panel = pd.concat(all_years, ignore_index=True)
    #Replace region codes with names
    panel['REGMCR'] = panel['REGMCR'].astype(str).str.zfill(2)  # Ensure codes are two digits
    panel['REGMCR'] = panel['REGMCR'].map(region_dict)
    panel_pivot = panel.pivot(index='REGMCR', columns='year', values='remote').reset_index()
    panel_pivot.to_csv(output_csv, index=False)
    return panel_pivot

In [ ]:
avg_pt_hours_panel = build_var_panel_from_dfs(
    dfs= df_list,
    years= years_t,
    var= "QC33MCR",
    codes = ['1','2','3','4','5','6','7','8'],
    var_codes = ['1','2','3','4'],
    output_csv='avg_pt_hours_panel.csv'
)

extra_work_panel = build_var_panel_from_dfs(
    dfs= df_list,
    years= years_t,
    var= "QB04",
    codes = ['1','2','3'],
    var_codes = ['1'],
    output_csv='extra_work_panel.csv'
)

did_not_work_panel = build_var_panel_from_dfs(
    dfs= df_list,
    years= years_t,
    var= "QB03",
    codes = ['1','2'],
    var_codes = ['1'],
    output_csv='did_not_work_panel.csv'
)

In [ ]:
reason_not_work_panel = build_var_panel_from_dfs(
    dfs= df_list,
    years= years_t,
    var= "QB03",
    codes = ['001','002','003','004','005','006','007','008','009','010','011','012','994','996','1','2','3','4','5','6','7','8','9','10'],
    var_codes = ['010','996','10'],
    output_csv='reason_not_work_panel.csv'
)